# RP-013 — Live Resume Parser API verification

Run this notebook only after starting Uvicorn in a separate Colab terminal. It sends real HTTP requests to the loaded Transformers model through FastAPI. All fixtures are synthetic; do not replace them with private resumes when saving outputs or artifacts.

Expected server command: `python -m uvicorn app.main:app --host 127.0.0.1 --port 8000 --workers 1`. Do not start another model manager from this notebook.

In [ ]:
from pathlib import Path
from io import BytesIO
import json
import os
import time

import requests

candidates = [Path('/content/ATS-CV/PDFPARSER'), Path.cwd(), Path.cwd() / 'PDFPARSER']
PROJECT_ROOT = next((path for path in candidates if (path / 'requirements-dev.txt').is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Clone the repository, then run this notebook from the PDFPARSER project')

os.chdir(PROJECT_ROOT)
BASE_URL = 'http://127.0.0.1:8000'
REQUEST_TIMEOUT_SECONDS = 600
ARTIFACT_DIR = PROJECT_ROOT / 'artifacts' / 'rp013'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print({'project_root': str(PROJECT_ROOT), 'base_url': BASE_URL})

## Test helpers

A `PASS` means the live response satisfied the intended HTTP and schema assertions. A `FAIL` is retained in `RESULTS` so the remaining cases can continue.

In [ ]:
RESULTS = []
PROFILE_FIELDS = {
    'personal_info', 'summary', 'work_experience', 'education',
    'skills', 'certifications', 'languages',
}

def assert_success_contract(body):
    assert body['status'] == 'success'
    assert body['schema_version'] == '1.0'
    assert set(body['data']) == PROFILE_FIELDS
    generation = body['metadata']['generation']
    assert generation['completion_status'] == 'completed'
    assert generation['input_tokens'] > 0
    assert generation['output_tokens'] > 0
    assert generation['generation_ms'] > 0
    assert generation['peak_allocated_vram_bytes'] > 0
    assert generation['peak_reserved_vram_bytes'] > 0

def assert_error_contract(body, expected_code):
    assert body['status'] == 'error'
    assert body['error']['code'] == expected_code
    assert isinstance(body['error']['message'], str)
    assert body['error']['message']

def has_value(values, expected):
    normalized = {str(value).casefold() for value in values}
    return expected.casefold() in normalized

def run_case(name, request_fn, expected_status, validator):
    started_at = time.perf_counter()
    response = None
    body = None
    try:
        response = request_fn()
        elapsed_ms = round((time.perf_counter() - started_at) * 1000, 2)
        body = response.json()
        assert response.status_code == expected_status, body
        validator(body)
        result = {
            'name': name, 'status': 'passed', 'http_status': response.status_code,
            'elapsed_ms': elapsed_ms, 'response': body,
        }
        print(f'PASS: {name} ({response.status_code}, {elapsed_ms} ms)')
    except Exception as exc:
        elapsed_ms = round((time.perf_counter() - started_at) * 1000, 2)
        result = {
            'name': name, 'status': 'failed',
            'http_status': response.status_code if response is not None else None,
            'elapsed_ms': elapsed_ms, 'error': f'{type(exc).__name__}: {exc}',
            'response': body,
        }
        print(f'FAIL: {name}: {result["error"]}')
    RESULTS.append(result)
    if body is not None:
        print(json.dumps(body, ensure_ascii=False, indent=2))
    return result

## 1. Liveness and real-model readiness

In [ ]:
def validate_health(body):
    assert body['status'] == 'ok'

def validate_ready(body):
    assert body['status'] == 'ready'
    assert body['inference_mode'] == 'transformers'
    assert body['model'] in {
        'Qwen/Qwen2.5-7B-Instruct',
        'Qwen/Qwen2.5-14B-Instruct',
    }

run_case(
    'health',
    lambda: requests.get(f'{BASE_URL}/api/v1/health', timeout=30),
    200,
    validate_health,
)
run_case(
    'ready',
    lambda: requests.get(f'{BASE_URL}/api/v1/ready', timeout=30),
    200,
    validate_ready,
)

## 2. English text resume

In [ ]:
ENGLISH_RESUME = '''
Nguyen Van A
Email: nguyenvana@example.com
Phone: 0901234567
Location: Ho Chi Minh City

Summary: Backend developer with three years of experience.

Work Experience:
Backend Developer, Example Company
January 2022 - December 2024
- Developed REST APIs using Python and FastAPI.
- Worked with PostgreSQL and Docker.

Education:
Bachelor of Computer Science, Example University, 2021

Skills: Python, FastAPI, PostgreSQL, Docker
Languages: Vietnamese, English
'''

def validate_english_resume(body):
    assert_success_contract(body)
    profile = body['data']
    assert profile['personal_info']['full_name'] == 'Nguyen Van A'
    assert profile['personal_info']['email'] == 'nguyenvana@example.com'
    assert has_value(profile['skills'], 'Python')
    assert has_value(profile['skills'], 'FastAPI')

run_case(
    'parse_text_english',
    lambda: requests.post(
        f'{BASE_URL}/api/v1/resumes/parse-text',
        json={'text': ENGLISH_RESUME},
        headers={'X-Request-ID': 'rp013_text_english'},
        timeout=REQUEST_TIMEOUT_SECONDS,
    ),
    200,
    validate_english_resume,
)

## 3. Vietnamese and English resume

In [ ]:
MULTILINGUAL_RESUME = '''
Trần Thị Bình
Email: binh.tran@example.com
Địa điểm: Đà Nẵng

Kinh nghiệm:
Kỹ sư phần mềm tại Công ty Example từ tháng 06/2022 đến nay.
Xây dựng REST API bằng Python và FastAPI.

Kỹ năng: Python, FastAPI, PostgreSQL
Languages: Tiếng Việt, English
'''

def validate_multilingual_resume(body):
    assert_success_contract(body)
    profile = body['data']
    assert profile['personal_info']['full_name'] == 'Trần Thị Bình'
    assert has_value(profile['skills'], 'Python')
    assert has_value(profile['skills'], 'FastAPI')

run_case(
    'parse_text_multilingual',
    lambda: requests.post(
        f'{BASE_URL}/api/v1/resumes/parse-text',
        json={'text': MULTILINGUAL_RESUME},
        headers={'X-Request-ID': 'rp013_text_multilingual'},
        timeout=REQUEST_TIMEOUT_SECONDS,
    ),
    200,
    validate_multilingual_resume,
)

## 4. Sparse resume

This case checks that the model uses null and empty collections instead of inventing a person or employment history.

In [ ]:
SPARSE_RESUME = 'Skills: Python, SQL'

def validate_sparse_resume(body):
    assert_success_contract(body)
    profile = body['data']
    assert profile['personal_info']['full_name'] is None
    assert profile['work_experience'] == []
    assert profile['education'] == []
    assert has_value(profile['skills'], 'Python')
    assert has_value(profile['skills'], 'SQL')

run_case(
    'parse_text_sparse',
    lambda: requests.post(
        f'{BASE_URL}/api/v1/resumes/parse-text',
        json={'text': SPARSE_RESUME},
        headers={'X-Request-ID': 'rp013_text_sparse'},
        timeout=REQUEST_TIMEOUT_SECONDS,
    ),
    200,
    validate_sparse_resume,
)

## 5. Synthetic PDF upload

In [ ]:
from tests.fixtures.documents import make_pdf_bytes

PDF_BYTES = make_pdf_bytes(
    'Le Minh C - Python Developer - Email: c.le@example.com - Skills: Python, FastAPI'
)

def validate_pdf_resume(body):
    assert_success_contract(body)
    assert has_value(body['data']['skills'], 'Python')

run_case(
    'parse_pdf',
    lambda: requests.post(
        f'{BASE_URL}/api/v1/resumes/parse',
        files={'file': ('synthetic_resume.pdf', PDF_BYTES, 'application/pdf')},
        headers={'X-Request-ID': 'rp013_pdf'},
        timeout=REQUEST_TIMEOUT_SECONDS,
    ),
    200,
    validate_pdf_resume,
)

## 6. Synthetic DOCX upload

In [ ]:
from docx import Document

document = Document()
document.add_heading('Phạm Minh D', level=1)
document.add_paragraph('Email: pham.minh.d@example.com')
document.add_paragraph('Vị trí: Lập trình viên Backend')
document.add_heading('Kinh nghiệm', level=2)
document.add_paragraph('Backend Developer tại Example Company từ 2023 đến nay.')
document.add_heading('Kỹ năng', level=2)
document.add_paragraph('Python, FastAPI, Docker')
DOCX_BUFFER = BytesIO()
document.save(DOCX_BUFFER)
DOCX_BYTES = DOCX_BUFFER.getvalue()

def validate_docx_resume(body):
    assert_success_contract(body)
    profile = body['data']
    assert profile['personal_info']['full_name'] == 'Phạm Minh D'
    assert has_value(profile['skills'], 'Python')

run_case(
    'parse_docx',
    lambda: requests.post(
        f'{BASE_URL}/api/v1/resumes/parse',
        files={
            'file': (
                'synthetic_resume.docx', DOCX_BYTES,
                'application/vnd.openxmlformats-officedocument.wordprocessingml.document',
            )
        },
        headers={'X-Request-ID': 'rp013_docx'},
        timeout=REQUEST_TIMEOUT_SECONDS,
    ),
    200,
    validate_docx_resume,
)

## 7. Request validation errors

These cases should be rejected before real generation and therefore should complete quickly.

In [ ]:
run_case(
    'reject_empty_text',
    lambda: requests.post(
        f'{BASE_URL}/api/v1/resumes/parse-text',
        json={'text': ''},
        headers={'X-Request-ID': 'rp013_empty'},
        timeout=30,
    ),
    422,
    lambda body: assert_error_contract(body, 'REQUEST_VALIDATION_ERROR'),
)

run_case(
    'reject_unsupported_file',
    lambda: requests.post(
        f'{BASE_URL}/api/v1/resumes/parse',
        files={'file': ('resume.txt', b'plain text', 'text/plain')},
        headers={'X-Request-ID': 'rp013_unsupported'},
        timeout=30,
    ),
    415,
    lambda body: assert_error_contract(body, 'UNSUPPORTED_DOCUMENT_TYPE'),
)

run_case(
    'reject_character_limit',
    lambda: requests.post(
        f'{BASE_URL}/api/v1/resumes/parse-text',
        json={'text': 'A' * 50_001},
        headers={'X-Request-ID': 'rp013_character_limit'},
        timeout=30,
    ),
    400,
    lambda body: assert_error_contract(body, 'INVALID_INPUT'),
)

## 8. Summary and artifact

Review semantic failures even when the HTTP contract passed. The Uvicorn terminal should contain exactly one `model_loaded` event for the entire run.

In [ ]:
passed = sum(result['status'] == 'passed' for result in RESULTS)
failed = len(RESULTS) - passed
summary = {
    'target': BASE_URL,
    'total': len(RESULTS),
    'passed': passed,
    'failed': failed,
    'results': RESULTS,
}

artifact_path = ARTIFACT_DIR / 'live_api_smoke.json'
artifact_path.write_text(
    json.dumps(summary, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

print(json.dumps({key: summary[key] for key in ('total', 'passed', 'failed')}, indent=2))
print(f'Artifact: {artifact_path}')
if failed:
    print('Some cases failed. Inspect RESULTS and the Uvicorn terminal before marking RP-013 complete.')
else:
    print('All live API cases passed. Stop Uvicorn with Ctrl+C and verify GPU memory release.')